In [5]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# Portable project-root detection
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

# If Jupyter is started from the "notebooks" folder,
# the project root is one level above it.
if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent

# If Jupyter is started from the project root,
# use the current directory directly.
elif (CURRENT_DIR / "notebooks").exists():
    PROJECT_DIR = CURRENT_DIR

# Otherwise, search upwards for the project root.
else:
    PROJECT_DIR = None

    for parent in [CURRENT_DIR] + list(CURRENT_DIR.parents):
        if (parent / "notebooks").exists() and (parent / "requirements.txt").exists():
            PROJECT_DIR = parent
            break

    if PROJECT_DIR is None:
        raise FileNotFoundError(
            "Project root could not be detected. "
            "Please start JupyterLab from the CyberXAI-CSE-IDS2018 project folder."
        )


# ============================================================
# Project directories
# ============================================================

RAW_DIR = PROJECT_DIR / "data" / "raw"
DOCUMENTATION_DIR = PROJECT_DIR / "documentation"

DOCUMENTATION_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Locate raw Parquet files
# ============================================================

parquet_files = sorted(RAW_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No Parquet files found in:\n{RAW_DIR}\n\n"
        "Download the 10 dataset files from the GitHub dataset-v1 release "
        "and place them inside data/raw/."
    )

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print(f"Parquet files found: {len(parquet_files)}")


# ============================================================
# Create dataset inventory
# ============================================================

inventory_records = []

for file_path in parquet_files:

    try:
        parquet_file = pq.ParquetFile(file_path)

        columns = parquet_file.schema.names

        label_column = "Label" if "Label" in columns else "Not found"

        inventory_records.append(
            {
                "file_name": file_path.name,
                "file_size_mb": round(
                    file_path.stat().st_size / (1024 ** 2), 2
                ),
                "rows": parquet_file.metadata.num_rows,
                "columns": len(columns),
                "row_groups": parquet_file.metadata.num_row_groups,
                "label_column": label_column,
                "read_status": "Read successfully",
            }
        )

    except Exception as error:

        inventory_records.append(
            {
                "file_name": file_path.name,
                "file_size_mb": round(
                    file_path.stat().st_size / (1024 ** 2), 2
                ),
                "rows": None,
                "columns": None,
                "row_groups": None,
                "label_column": None,
                "read_status": f"Read failed: {error}",
            }
        )


# ============================================================
# Display inventory
# ============================================================

inventory_df = pd.DataFrame(inventory_records)

display(inventory_df)


# ============================================================
# Summary
# ============================================================

total_files = len(inventory_df)
total_rows = inventory_df["rows"].fillna(0).sum()

print()
print(f"Total files: {total_files}")
print(f"Total rows: {int(total_rows):,}")


# ============================================================
# Save inventory
# ============================================================

inventory_path = DOCUMENTATION_DIR / "dataset_inventory.csv"

inventory_df.to_csv(
    inventory_path,
    index=False
)

print(f"Inventory saved to: {inventory_path}")

Project directory: D:\Sami Data Set\CyberXAI-CSE-IDS2018
Raw data directory: D:\Sami Data Set\CyberXAI-CSE-IDS2018\data\raw
Parquet files found: 10


,file_name,file_size_mb,rows,columns,row_groups,label_column,read_status
0,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,79.61,771587,78,1,Label,Read successfully
1,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,72.89,619346,78,1,Label,Read successfully
2,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,85.35,954846,78,1,Label,Read successfully
3,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,31.90,561396,78,1,Label,Read successfully
4,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,94.05,794812,78,1,Label,Read successfully
5,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,55.31,591873,78,1,Label,Read successfully
6,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,48.59,456873,78,1,Label,Read successfully
7,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,27.93,249170,78,1,Label,Read successfully
8,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,99.14,830224,78,1,Label,Read successfully
9,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,96.69,829405,78,1,Label,Read successfully



Total files: 10
Total rows: 6,659,532
Inventory saved to: D:\Sami Data Set\CyberXAI-CSE-IDS2018\documentation\dataset_inventory.csv
